In [22]:
import requests
import pandas as pd
import json

In [29]:
df = pd.read_csv('./qna_data.csv')
df.head()

,Question,Source Docs,Question Type,Source Chunk Type,Answer
0,How has Apple's total net sales changed over t...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, Apple's total..."
1,What are the major factors contributing to the...,*AAPL*,Multi-Doc RAG,Text,In the most recent 10-Q for the quarter ended ...
2,Has there been any significant change in Apple...,*AAPL*,Multi-Doc RAG,Table,"Yes, there has been a change in Apple's operat..."
3,How has Apple's revenue from iPhone sales fluc...,*AAPL*,Multi-Doc RAG,Table,The revenue from iPhone sales for Apple has fl...
4,Can any trends be identified in Apple's Servic...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, there is a tr..."


In [50]:
import re

def extract_sources(text: str):
    # Берём всё после SOURCE(S):
    match = re.search(r"SOURCE\(S\):\s*(.+)$", text, flags=re.MULTILINE | re.DOTALL)
    if not match:
        return []

    sources_raw = match.group(1).strip()

    # Разбиваем по запятым
    parts = [p.strip() for p in sources_raw.split(",") if p.strip()]

    cleaned_sources = []
    for p in parts:
        # убираем кавычки, скобки, точки в конце
        p = re.sub(r'^[\s"\(\[\{]+', "", p)   # слева
        p = re.sub(r'[\s"\)\]\}\.]+$', "", p) # справа

        # убираем лишние пробелы внутри
        p = re.sub(r"\s+", " ", p).strip()

        if p:
            cleaned_sources.append(p)

    return cleaned_sources

In [51]:
df['sources'] = df['Answer'].apply(extract_sources)

In [56]:
df

,Question,Source Docs,Question Type,Source Chunk Type,Answer,sources
0,How has Apple's total net sales changed over t...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, Apple's total...","[2022 Q3 AAPL.pdf, 2023 Q1 AAPL.pdf, 2023 Q2 A..."
1,What are the major factors contributing to the...,*AAPL*,Multi-Doc RAG,Text,In the most recent 10-Q for the quarter ended ...,[2023 Q3 AAPL.pdf]
2,Has there been any significant change in Apple...,*AAPL*,Multi-Doc RAG,Table,"Yes, there has been a change in Apple's operat...","[2022 Q3 AAPL.pdf, 2023 Q1 AAPL.pdf, 2023 Q2 A..."
3,How has Apple's revenue from iPhone sales fluc...,*AAPL*,Multi-Doc RAG,Table,The revenue from iPhone sales for Apple has fl...,"[2022 Q3 AAPL.pdf, 2023 Q1 AAPL.pdf, 2023 Q2 A..."
4,Can any trends be identified in Apple's Servic...,*AAPL*,Multi-Doc RAG,Table,"Based on the provided documents, there is a tr...","[2022 Q3 AAPL.pdf, 2023 Q1 AAPL.pdf, 2023 Q2 A..."
...,...,...,...,...,...,...
190,"For Amazon's Q1 2023 10-Q, align the details o...",*2023 Q1 AMZN*,Single-Doc Multi-Chunk RAG,Table,"In Amazon's Q1 2023 10-Q, the details of debt ...",[2023 Q1 AMZN.pdf]
191,Analyze how Amazon's effective tax rate report...,*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The effective tax rate for Amazon as reported ...,[2023 Q3 AMZN.pdf]
192,"From Amazon's Q3 2023 10-Q, how does the opera...",*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The operational expenses section in Amazon's Q...,[2023 Q3 AMZN.pdf]
193,"In the latest 10-Q, how does the revenue from ...",*2023 Q3 AMZN*,Single-Doc Multi-Chunk RAG,Table,The latest 10-Q does not provide specific info...,[2023 Q3 AMZN.pdf]


In [19]:
# http_ = 'http://localhost:9621/query/data'
# data = {"query": "machine learning algorithms"}

# r = requests.post(url=http_, json=data)

# r.status_code

In [18]:
# import ollama

# response = ollama.embed(
#     model='bge-m3',
#     input='The sky is blue because of Rayleigh scattering',
# )
# print(response.embeddings)

In [64]:
example = {
  "response": "Artificial Intelligence (AI) is a branch of computer science that aims to create intelligent machines capable of performing tasks that typically require human intelligence, such as learning, reasoning, and problem-solving.",
  "references": [
    {
      "reference_id": "1",
      "file_path": "/documents/ai_overview.pdf"
    },
    {
      "reference_id": "2",
      "file_path": "/documents/machine_learning.txt"
    }
  ]
}

In [ ]:
def parse_answer(answer: str):
    js_example = json.loads(json.dumps(answer))
    return js_example['response'], [i['file_path'] for i in js_example['references']]

In [75]:
parse_answer(example)

('Artificial Intelligence (AI) is a branch of computer science that aims to create intelligent machines capable of performing tasks that typically require human intelligence, such as learning, reasoning, and problem-solving.',
 ['/documents/ai_overview.pdf', '/documents/machine_learning.txt'])

## Генерация ошибочных датасетов

In [1]:
import os
import pandas as pd
import getpass
from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole

# Просим ввести строку для подключения гигачата, если её нет в переменных окружения
if "GIGACHAT_CREDENTIALS" not in os.environ:
    os.environ["GIGACHAT_CREDENTIALS"] = getpass.getpass("GigaChat Credentials: ")

if "GIGACHAT_SCOPE" not in os.environ:
    os.environ["GIGACHAT_SCOPE"] = getpass.getpass("GigaChat Scope: ")

In [2]:
df = pd.read_csv('./qna_data.csv')

In [44]:
payload = Chat(
    messages=[
        Messages(
            role=MessagesRole.SYSTEM,
            content="""
You are a text corruption engine. Your only task is to introduce spelling noise.

You are given an input text. Output a corrupted version with realistic spelling mistakes.

STRICT RULES:
1) Do NOT add, remove, or reorder words.
2) Do NOT change punctuation, formatting, or line breaks.
3) Do NOT change letter casing.
4) Do NOT modify any numbers, dates, currency values, or percentages.
5) Do NOT modify proper names, abbreviations, or words containing apostrophes.
6) Do NOT modify month names (January ... December).
7) Do NOT modify any word of length <= 5.
8) Do NOT modify any line containing "SOURCE:" or "SOURCE(S):", and do NOT modify anything after "SOURCE(S):".
9) Do NOT modify file names containing ".pdf".
10) Corrupt about 20% of words with length >= 5.
11) For each corrupted word apply exactly ONE operation:
   - delete one character
   - replace one character
   - swap two adjacent characters
12) NEVER insert spaces inside words. NEVER split words.
13) Do NOT shorten words (e.g., million -> mln is forbidden).

OUTPUT FORMAT (STRICT):
Return ONLY the corrupted text. No comments, no explanations, no extra symbols.
"""
        )
    ],
    temperature=0.1
)

In [45]:
# Реализуем функцию для отправки запроса в GigaChat
def ask_gigachat(text: str) -> str:
    with GigaChat(verify_ssl_certs=False, model="GigaChat-Max") as giga:
        # Добавляем системное сообщение с правилами и пользовательское сообщение с текстом
        local_payload = Chat(messages=list(payload.messages), temperature=0.7)
        # local_payload.messages.append(Messages(role=MessagesRole.SYSTEM, content=prompt))
        local_payload.messages.append(Messages(role=MessagesRole.USER, content=text))
        response = giga.chat(local_payload)
        return response.choices[0].message.content

### Орфографические ошибки

In [46]:

print(f"Реальный текст: {df['Answer'][0]}")

print(f"Ответ: {ask_gigachat(text=df['Answer'][0])}")

Реальный текст: Based on the provided documents, Apple's total net sales have changed over time as follows:

- For the quarterly period ended June 25, 2022, the total net sales were $82,959 million. (SOURCE: 2022 Q3 AAPL.pdf)
- For the quarterly period ended December 31, 2022, the total net sales were $117,154 million. (SOURCE: 2023 Q1 AAPL.pdf)
- For the quarterly period ended April 1, 2023, the total net sales were $94,836 million. (SOURCE: 2023 Q2 AAPL.pdf)
- For the quarterly period ended July 1, 2023, the total net sales were $81,797 million. (SOURCE: 2023 Q3 AAPL.pdf)

From these figures, it can be observed that there was an increase in total net sales from the quarter ended June 25, 2022, to the quarter ended December 31, 2022. However, there was a subsequent decrease in total net sales in the quarters ended April 1, 2023, and July 1, 2023.

SOURCE(S): 2022 Q3 AAPL.pdf, 2023 Q1 AAPL.pdf, 2023 Q2 AAPL.pdf, 2023 Q3 AAPL.pdf
Ответ: Baced on teh provded documnts, Apple's totla net s

In [7]:
# 27788